# Implement LoRA on a Linear Layer

**Difficulty**: 🟡 Medium

**Companies**: Meta, Google, Anthropic, OpenAI, Databricks

---

### Problem Statement

Low-Rank Adaptation (LoRA) is one of the most important parameter-efficient fine-tuning techniques in modern deep learning. Instead of fine-tuning all parameters of a pretrained model, LoRA freezes the original weights and injects small, trainable low-rank matrices into each layer. This dramatically reduces the number of trainable parameters while maintaining model quality.

The key idea: instead of learning a full weight update ΔW (d_in × d_out), LoRA decomposes it into two low-rank matrices: ΔW = B × A, where A is (d_in, r) and B is (r, d_out), with rank r << min(d_in, d_out).

The forward pass becomes: output = Wx + (α/r) × x @ A @ B

Your task is to implement LoRA from scratch, apply it to an existing model, and merge the adapted weights back.

---

### Requirements

1. **LoRALinear** — A module that wraps an existing `nn.Linear`, adds low-rank matrices A and B, and freezes the original weights.
2. **apply_lora** — A function that injects LoRA into specified layers of an existing model.
3. **merge_lora** — A function that merges LoRA weights back into the original linear layer (W_new = W + (α/r) × BᵀAᵀ transposed appropriately).
4. **Parameter Efficiency** — Show that trainable parameters are much fewer than total parameters.
5. **Training** — Train a simple model with LoRA on synthetic data.

---

### Constraints

- ✅ A should be initialized with Kaiming uniform; B should be initialized to zeros (so ΔW starts at zero).
- ✅ Original weights must be frozen (requires_grad=False).
- ✅ Merged model must produce identical outputs to the LoRA model.
- ❌ Do **not** use any existing LoRA libraries (peft, loralib, etc.).

---

<details>
  <summary>💡 Hint</summary>

  - Initialize A with `nn.init.kaiming_uniform_` and B with zeros. This ensures the LoRA contribution starts at zero before training.
  - The scaling factor is `alpha / rank`. This controls the magnitude of the LoRA update relative to the original weights.
  - When merging, add `(alpha/r) * (B.T @ A.T).T` to the original weight, or equivalently `(alpha/r) * A.T @ B.T` reshaped to match the weight.
  - To apply LoRA to a model, iterate through named modules and replace matching `nn.Linear` layers with `LoRALinear` wrappers.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

In [ ]:
# A simple pretrained model to apply LoRA to
class SimpleModel(nn.Module):
    def __init__(self, input_dim=64, hidden_dim=128, output_dim=10):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

torch.manual_seed(42)
model = SimpleModel()
print("Model architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
class LoRALinear(nn.Module):
    """
    Wraps an existing nn.Linear layer with low-rank adaptation.
    
    Forward: output = W @ x + (alpha / r) * (x @ A) @ B
    where A is (d_in, r) and B is (r, d_out).
    Original weights W are frozen.
    """
    def __init__(self, linear: nn.Linear, rank: int = 4, alpha: float = 1.0):
        super().__init__()
        self.linear = linear
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank

        d_in = linear.in_features
        d_out = linear.out_features

        # TODO: Freeze original linear layer weights (set requires_grad to False)
        ...

        # TODO: Create low-rank matrices A (d_in, rank) and B (rank, d_out) as nn.Parameter
        # Initialize A with kaiming_uniform_ and B with zeros
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass: original output + scaled low-rank adaptation.
        output = linear(x) + (alpha/r) * (x @ A) @ B
        """
        # TODO: Compute original linear output
        # TODO: Compute LoRA contribution: (x @ A) @ B * scaling
        # TODO: Return sum of both
        ...


def apply_lora(model: nn.Module, rank: int = 4, alpha: float = 1.0,
               target_modules: list = None) -> nn.Module:
    """
    Replace target nn.Linear layers in model with LoRALinear wrappers.
    
    Args:
        model: The model to modify.
        rank: LoRA rank.
        alpha: LoRA scaling factor.
        target_modules: List of module name substrings to target (e.g., ['fc1', 'fc2']).
    
    Returns:
        Modified model with LoRA layers.
    """
    # TODO: Iterate through named modules
    # For each module that matches target_modules and is nn.Linear,
    # replace it with a LoRALinear wrapper using setattr
    ...


def merge_lora(model: nn.Module) -> nn.Module:
    """
    Merge LoRA weights back into the original linear layers.
    W_new = W_old + (alpha/r) * A @ B
    Replace LoRALinear modules with plain nn.Linear using merged weights.
    
    Returns:
        Model with merged weights (no more LoRA modules).
    """
    # TODO: Find all LoRALinear modules
    # For each, compute merged weight: W + scaling * (A @ B).T
    # Create a new nn.Linear with the merged weight and original bias
    # Replace the LoRALinear module with the new nn.Linear
    ...


def count_parameters(model: nn.Module):
    """Count total and trainable parameters."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [ ]:
# === Validation ===
torch.manual_seed(42)

# Create model and apply LoRA
model = SimpleModel()
lora_model = copy.deepcopy(model)
apply_lora(lora_model, rank=4, alpha=1.0, target_modules=['fc1', 'fc2'])

# 1. Parameter count comparison
print("=== Parameter Count Test ===")
total_orig, train_orig = count_parameters(model)
total_lora, train_lora = count_parameters(lora_model)
print(f"Original model: {total_orig:,} total, {train_orig:,} trainable")
print(f"LoRA model:     {total_lora:,} total, {train_lora:,} trainable")
print(f"Trainable ratio: {train_lora/total_lora*100:.1f}%")
assert train_lora < train_orig, "LoRA should have fewer trainable params!"
print("PASSED\n")

# 2. Train with LoRA on synthetic data
print("=== Training Test ===")
X = torch.randn(200, 64)
y = torch.randint(0, 10, (200,))
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, lora_model.parameters()), lr=1e-3)

losses = []
for epoch in range(50):
    optimizer.zero_grad()
    logits = lora_model(X)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Initial loss: {losses[0]:.4f}")
print(f"Final loss:   {losses[-1]:.4f}")
assert losses[-1] < losses[0], "Loss should decrease during training!"
print("PASSED\n")

# 3. Merge and compare outputs
print("=== Merge Test ===")
test_input = torch.randn(10, 64)
lora_model.eval()
with torch.no_grad():
    lora_output = lora_model(test_input)

merged_model = copy.deepcopy(lora_model)
merge_lora(merged_model)
merged_model.eval()
with torch.no_grad():
    merged_output = merged_model(test_input)

print(f"Max difference: {(lora_output - merged_output).abs().max().item():.2e}")
assert torch.allclose(lora_output, merged_output, atol=1e-5), "Merged model output differs from LoRA model!"
print("PASSED\n")

# 4. Verify merged model has no LoRA modules
print("=== Clean Merge Test ===")
has_lora = any(isinstance(m, LoRALinear) for m in merged_model.modules())
assert not has_lora, "Merged model should not contain LoRALinear modules!"
print("No LoRALinear modules remain after merge.")
print("PASSED\n")

print("All tests passed!")